# LeetCode 513 — Find Bottom Left Tree Value bằng Python

Link đề: https://leetcode.com/problems/find-bottom-left-tree-value/ · Độ khó: **Medium** · Chủ đề: Tree, Binary Tree, Breadth-First Search, Depth-First Search

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def findBottomLeftValue(self, root: Optional[TreeNode]) -> int:
        ...
```

Ghi chú: notebook tự dựng `TreeNode` và hàm `build_tree` dựng cây từ list kiểu LeetCode (`[1,2,3,4,null,5,6,null,null,7]`). Các ví dụ thêm do notebook tự đặt (có đáp án đã kiểm chứng bằng code); nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm giá trị của nút **trái nhất ở hàng sâu nhất** (hàng cuối cùng) của cây nhị phân.
2. Nắm các công cụ Python: định nghĩa `class TreeNode`, dựng cây từ list có `None`, `collections.deque` cho BFS, đệ quy DFS, `nonlocal`, ngăn xếp tường minh, trả về **nhiều giá trị** (tuple) từ hàm đệ quy.
3. Đi từ **liệt kê từng tầng rồi lấy phần tử đầu của tầng cuối** → **BFS duyệt phải-trước-trái, nút cuối cùng lấy ra là đáp án** → **DFS tiền tự theo dõi độ sâu lớn nhất** → **DFS từ dưới lên trả về `(chiều cao, giá trị)`** → **DFS lặp bằng ngăn xếp**.
4. Hiểu hai mấu chốt: (a) "hàng cuối" nghĩa là **độ sâu lớn nhất**, không phải "đi hoài về bên trái"; (b) trong DFS tiền tự **trái trước**, nút **đầu tiên** chạm tới một độ sâu mới là nút trái nhất của độ sâu đó.
5. Tránh các lỗi kinh điển: chỉ đi theo `left` mãi, cập nhật bằng `>=` (ra nút phải nhất), khởi tạo độ sâu tối đa sai (cây một nút), BFS trả về nút phải nhất, và quên chốt kích thước tầng trước khi duyệt.

## 2. Đề bài & ví dụ

Cho nút gốc `root` của một cây nhị phân. Hãy trả về **giá trị nút trái nhất** ở **hàng cuối cùng** (hàng sâu nhất) của cây.

**Ví dụ 1**

- Input: `root = [2,1,3]`
- Output: `1`
- Giải thích: hàng cuối là `[1, 3]`, nút trái nhất có giá trị `1`.

**Ví dụ 2**

- Input: `root = [1,2,3,4,null,5,6,null,null,7]`
- Output: `7`
- Giải thích: hàng cuối chỉ có nút `7`.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `root = [1,2,3,null,4,5]`
- Output: `4`
- Giải thích: các hàng là `[1]`, `[2, 3]`, `[4, 5]`. Hàng cuối là `[4, 5]`: nút `4` (con **phải** của `2`) nằm bên trái nút `5` (con trái của `3`), nên nút trái nhất là `4`.

**Ràng buộc:**

- Số nút của cây nằm trong khoảng `[1, 10^4]`.
- `-2^31 <= Node.val <= 2^31 - 1`

Chú ý: "trái nhất" nói về **vị trí trong hàng**, **không** có nghĩa nút đó phải là con trái của cha. Ví dụ 3 cho thấy đáp án `4` là con **phải** của nút `2`.

In [1]:
from typing import Optional, List
from collections import deque
import random
import sys

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([2, 1, 3], 1),
    ([1, 2, 3, 4, None, 5, 6, None, None, 7], 7),
    ([1, 2, 3, None, 4, 5], 4),
]
for vals, expected in examples:
    assert tree_to_list(build_tree(vals)) == vals
    print(vals, "-> kỳ vọng", expected)

[2, 1, 3] -> kỳ vọng 1
[1, 2, 3, 4, None, 5, 6, None, None, 7] -> kỳ vọng 7
[1, 2, 3, None, 4, 5] -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Nút cây, độ sâu (depth) và hàng (level)

Mỗi nút `TreeNode` có `val`, `left`, `right`. **Độ sâu** của một nút là số cạnh từ gốc xuống nút đó: gốc có độ sâu `0`, con của gốc độ sâu `1`, ... Các nút cùng độ sâu tạo thành một **hàng (level)**. "Hàng cuối" là hàng có độ sâu lớn nhất.

In [2]:
def show_levels(root):
    level, d = [root], 0
    while any(x is not None for x in level):
        print(f"tầng {d}:", [x.val if x else None for x in level])
        level = [c for x in level if x for c in (x.left, x.right)]
        d += 1

t = build_tree([1, 2, 3, 4, None, 5, 6, None, None, 7])
show_levels(t)

tầng 0: [1]
tầng 1: [2, 3]
tầng 2: [4, None, 5, 6]
tầng 3: [None, None, 7, None, None, None]


Cây trên:

```
        1
      /   \
     2     3
    /     / \
   4     5   6
        /
       7
```

Nhìn ra ngay: chiều sâu lớn nhất là `3` (nút `7`), hàng cuối chỉ có một nút.

### 3.2. Dựng cây từ list kiểu LeetCode

Danh sách `[1,2,3,4,None,5,6,None,None,7]` liệt kê **theo từng hàng, từ trái sang phải**, với `None` đánh dấu chỗ trống. `build_tree` (ở cell đầu) dùng một hàng đợi: lấy nút đang chờ, gán con trái rồi con phải từ list, rồi đưa các con thật vào hàng đợi. `tree_to_list` làm ngược lại để ta kiểm tra việc dựng đã đúng.

In [3]:
print(tree_to_list(build_tree([1, 2, 3, None, 4, 5])))
r = build_tree([1, 2, 3, None, 4, 5])
print("con phải của nút 2:", r.left.right.val)
print("con trái của nút 3:", r.right.left.val)

[1, 2, 3, None, 4, 5]
con phải của nút 2: 4
con trái của nút 3: 5


### 3.3. `deque` và BFS (duyệt theo tầng)

`collections.deque` là hàng đợi hai đầu: `append` thêm vào cuối, `popleft` lấy ở đầu với chi phí `O(1)` (còn `list.pop(0)` tốn `O(n)`). **BFS** dùng hàng đợi để duyệt cây từng hàng một.

Mẹo quan trọng: ở đầu mỗi vòng, **chốt `size = len(queue)`** — đó là số nút của hàng hiện tại; xử lý đúng `size` nút này, các con được thêm vào sẽ thuộc hàng kế.

In [4]:
def bfs_levels_demo(root):
    queue = deque([root])
    level = 0
    while queue:
        size = len(queue)                   # chốt số nút của hàng hiện tại
        row = []
        for _ in range(size):
            node = queue.popleft()
            row.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        print("hàng", level, ":", row)
        level += 1

bfs_levels_demo(build_tree([1, 2, 3, 4, None, 5, 6, None, None, 7]))

hàng 0 : [1]
hàng 1 : [2, 3]
hàng 2 : [4, 5, 6]
hàng 3 : [7]


### 3.4. DFS tiền tự (preorder) và độ sâu

**DFS** (duyệt theo chiều sâu) viết bằng đệ quy: thăm nút rồi gọi lên con trái, sau đó con phải (**tiền tự**, "nút → trái → phải"). Ta truyền thêm tham số `depth` (độ sâu hiện tại): gọi con thì `depth + 1`. Thứ tự "trái trước phải sau" rất quan trọng cho bài này: trong số các nút cùng độ sâu, nút **trái nhất** luôn được gặp **trước**.

In [5]:
def preorder_with_depth(node, depth, out):
    if node is None:
        return
    out.append((node.val, depth))
    preorder_with_depth(node.left, depth + 1, out)
    preorder_with_depth(node.right, depth + 1, out)

out = []
preorder_with_depth(build_tree([1, 2, 3, 4, None, 5, 6, None, None, 7]), 0, out)
print(out)

[(1, 0), (2, 1), (4, 2), (3, 1), (5, 2), (7, 3), (6, 2)]


Quan sát: thứ tự thăm `1, 2, 4, 3, 5, 7, 6`. Trong các nút độ sâu 2 (`4, 5, 6`), nút `4` được gặp trước — và nó là nút trái nhất ở độ sâu 2. Nhưng độ sâu lớn nhất là `3` (nút `7`).

### 3.5. `nonlocal` và hàm đệ quy lồng nhau

Hàm `dfs` lồng trong hàm chính có thể **đọc** biến ngoài; để **gán lại** biến (ví dụ `best = ...`) cần khai báo `nonlocal best`.

In [6]:
def demo():
    count = 0
    def visit():
        nonlocal count
        count += 1
    for _ in range(3):
        visit()
    return count

print(demo())

3


### 3.6. Trả về nhiều giá trị bằng tuple

Một hàm đệ quy có thể trả về **cặp** `(chiều cao, giá trị)`. Phía gọi tách bằng `h, v = dfs(child)`. Ta sẽ dùng ở cách 4. Ví dụ nhỏ: hàm trả về `(độ sâu lớn nhất, giá trị nút sâu nhất đầu tiên)` của một chuỗi.

In [7]:
def min_and_max(xs):
    return min(xs), max(xs)

lo, hi = min_and_max([4, 9, 1, 7])
print(lo, hi)

1 9


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Bài hỏi hai điều: **hàng nào là cuối** và **nút nào trái nhất trong hàng đó**.

Thử một cách "ngây thơ": cứ đi theo `left` mãi cho tới khi hết. Với cây `[1,2,3,4,None,5,6,None,None,7]` ta đi `1 → 2 → 4` và dừng ở `4` — nhưng đáp án là `7`! Lý do: `4` ở độ sâu 2, còn `7` ở độ sâu 3 (đi qua nhánh **phải** của gốc). Vậy **nhánh trái có thể nông hơn nhánh phải**.

Ví dụ khác: `[1,2,3,None,4,5]`: các hàng là `[1]`, `[2,3]`, `[4,5]`. Hàng cuối `[4,5]` → trái nhất là `4`, mặc dù `4` là con **phải** của `2`.

Cách nghĩ tự nhiên: **liệt kê từng hàng**, lấy **hàng cuối**, lấy **phần tử đầu** của nó. Hãy làm đúng như vậy bằng tay cho `[1,2,3,4,None,5,6,None,None,7]`:

| Hàng | Các nút từ trái sang phải |
|---|---|
| 0 | `[1]` |
| 1 | `[2, 3]` |
| 2 | `[4, 5, 6]` |
| 3 | `[7]` |

Hàng cuối là hàng 3 → trái nhất = `7`.

In [8]:
tree = build_tree([1, 2, 3, None, 4, 5])
show_levels(tree)
print("hàng cuối là hàng 2, phần tử đầu là 4")

tầng 0: [1]
tầng 1: [2, 3]
tầng 2: [None, 4, 5, None]
hàng cuối là hàng 2, phần tử đầu là 4


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi hàng rồi lấy `levels[-1][0]`

**Ý tưởng:**

1. Dùng BFS theo tầng để tạo `levels`: list gồm các list giá trị của từng hàng.
2. Hàng cuối là `levels[-1]`; trái nhất là phần tử đầu `levels[-1][0]`.

Cách này "dịch nguyên văn đề bài" nên rất khó sai. Tốn bộ nhớ vì lưu mọi giá trị.

In [9]:
def bottom_left_levels(root):
    levels = []
    queue = deque([root])
    while queue:
        row = []
        for _ in range(len(queue)):
            node = queue.popleft()
            row.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        levels.append(row)
    return levels[-1][0]

for vals, expected in examples:
    got = bottom_left_levels(build_tree(vals))
    print(vals, "->", got)
    assert got == expected

[2, 1, 3] -> 1
[1, 2, 3, 4, None, 5, 6, None, None, 7] -> 7
[1, 2, 3, None, 4, 5] -> 4


**Độ phức tạp.**

- **Time:** `O(n)` — mỗi nút vào/ra hàng đợi một lần.
- **Space:** `O(n)` — `levels` chứa toàn bộ `n` giá trị (hàng đợi tối đa bằng độ rộng cây).

**Nút thắt:** ta lưu **mọi** hàng trong khi chỉ cần **phần tử đầu của hàng cuối**. Có thể bỏ `levels` đi và chỉ giữ một biến.

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát giúp bỏ bớt bộ nhớ và code:

1. **BFS theo tầng: chỉ cần nhớ phần tử đầu của hàng hiện tại.** Mỗi khi bắt đầu một hàng mới, ghi `ans = giá trị nút đầu hàng`. Khi hàng đợi cạn thì `ans` là phần tử đầu của hàng cuối.
2. **BFS "phải trước, trái sau":** nếu ta thêm con **phải** vào hàng đợi **trước** con trái, thì thứ tự lấy ra là "từ phải sang trái" theo từng hàng, và theo hàng từ nông tới sâu. Vậy **nút cuối cùng được lấy ra** chính là nút **trái nhất của hàng cuối**. Không cần đếm hàng.
3. **DFS tiền tự trái trước:** trong số các nút cùng độ sâu, nút trái nhất được thăm **đầu tiên**. Do đó ta duyệt, và mỗi lần gặp một nút có **độ sâu lớn hơn hẳn** độ sâu lớn nhất đã thấy, ta ghi nhận `ans = node.val`. Dùng `>` (không phải `>=`) để các nút "cùng độ sâu nhưng bên phải" **không** ghi đè.
4. **DFS từ dưới lên:** mỗi nút trả về `(chiều cao cây con, giá trị trái nhất của hàng cuối trong cây con đó)`. Cha chọn con **sâu hơn hẳn**; hòa thì chọn con **trái**.

Cả bốn cách đều `O(n)` thời gian; khác nhau ở bộ nhớ và cách nghĩ.

## 7. Các cách tiếp theo

### Cách 2 — BFS phải-trước-trái, nút cuối cùng lấy ra là đáp án

**Giải thích từng đoạn:**

1. Hàng đợi chứa `root`.
2. Mỗi vòng `popleft` một nút `node`; thêm `node.right` rồi `node.left` (nếu có) vào hàng đợi — **phải trước**.
3. Khi hàng đợi cạn, `node` cuối cùng là đáp án (trái nhất của hàng cuối).

Vì sao đúng: thứ tự lấy ra là hàng 0, hàng 1, ... và **trong mỗi hàng đi từ phải qua trái**. Nút cuối cùng nằm ở hàng sâu nhất, và là nút **ngoài cùng bên trái** của hàng đó.

In [10]:
def bottom_left_bfs(root):
    queue = deque([root])
    node = root
    while queue:
        node = queue.popleft()
        if node.right:
            queue.append(node.right)
        if node.left:
            queue.append(node.left)
    return node.val

for vals, expected in examples:
    got = bottom_left_bfs(build_tree(vals))
    assert got == expected
print("OK cách 2 trên ví dụ")

OK cách 2 trên ví dụ


In [11]:
class Solution:
    def findBottomLeftValue(self, root: Optional[TreeNode]) -> int:
        queue = deque([root])
        node = root
        while queue:
            node = queue.popleft()
            if node.right:
                queue.append(node.right)
            if node.left:
                queue.append(node.left)
        return node.val

print([Solution().findBottomLeftValue(build_tree(v)) for v, _ in examples])

[1, 7, 4]


**Độ phức tạp (cách 2):** Time `O(n)`; Space `O(w)` với `w` là độ rộng lớn nhất của cây (`w ≤ n`). Chỉ vài dòng và không cần biến đếm hàng.

### Cách 3 — DFS tiền tự, theo dõi độ sâu lớn nhất

**Giải thích từng đoạn:**

1. Biến `best_depth = -1` (chưa thấy nút nào) và `ans`.
2. `dfs(node, depth)`: nếu `node` rỗng thì thoát. Nếu `depth > best_depth` → đây là **nút đầu tiên** chạm độ sâu mới: `best_depth = depth`, `ans = node.val`.
3. Gọi `dfs(node.left, depth + 1)` **trước**, rồi `dfs(node.right, depth + 1)`.

Khởi tạo `best_depth = -1` (không phải `0`) để **chính gốc** (độ sâu `0`) cũng được ghi nhận, tránh sai ở cây một nút.

In [12]:
def bottom_left_dfs(root):
    best_depth, ans = -1, None

    def dfs(node, depth):
        nonlocal best_depth, ans
        if node is None:
            return
        if depth > best_depth:          # nút đầu tiên (trái nhất) chạm độ sâu mới
            best_depth, ans = depth, node.val
        dfs(node.left, depth + 1)       # TRÁI trước
        dfs(node.right, depth + 1)

    dfs(root, 0)
    return ans

for vals, expected in examples:
    got = bottom_left_dfs(build_tree(vals))
    assert got == expected
print("OK cách 3 trên ví dụ")

OK cách 3 trên ví dụ


In [13]:
class Solution:
    def findBottomLeftValue(self, root: Optional[TreeNode]) -> int:
        best_depth, ans = -1, root.val

        def dfs(node: Optional[TreeNode], depth: int) -> None:
            nonlocal best_depth, ans
            if node is None:
                return
            if depth > best_depth:
                best_depth, ans = depth, node.val
            dfs(node.left, depth + 1)
            dfs(node.right, depth + 1)

        dfs(root, 0)
        return ans

print([Solution().findBottomLeftValue(build_tree(v)) for v, _ in examples])

[1, 7, 4]


**Độ phức tạp (cách 3):** Time `O(n)`; Space `O(h)` cho ngăn xếp đệ quy (`h` là chiều cao cây, tối đa `n`). Với cây lệch `10^4` nút, đệ quy sâu `10^4` — trên LeetCode thường vẫn qua, nhưng ở máy cá nhân cần nới `sys.setrecursionlimit` hoặc dùng cách 5.

### Cách 4 — DFS từ dưới lên, trả về `(chiều cao, giá trị)`

**Ý tưởng khác:** cho hàm đệ quy mô tả **cây con**: `dfs(node)` trả về `(h, v)` với `h` là số nút trên đường dài nhất xuống lá (chiều cao tính bằng số nút) và `v` là giá trị trái nhất của hàng cuối **trong cây con đó**.

- Nút rỗng: `(0, None)`.
- Lá: `(1, node.val)`.
- Nút có hai con: lấy `(hl, vl)` và `(hr, vr)`. Nếu `hr > hl` thì hàng cuối của cây con nằm ở nhánh phải → `(hr + 1, vr)`; ngược lại (kể cả **hòa**) nhánh trái → `(hl + 1, vl)`. Nếu cả hai con rỗng → `(1, node.val)`.

Chú ý khi một con rỗng: `h = 0` nhỏ hơn con kia nên tự nhiên chọn con còn lại.

In [14]:
def bottom_left_dp(root):
    def dfs(node):
        if node is None:
            return 0, None
        hl, vl = dfs(node.left)
        hr, vr = dfs(node.right)
        if hl == 0 and hr == 0:                 # lá
            return 1, node.val
        if hr > hl:                             # nhánh phải sâu hơn hẳn
            return hr + 1, vr
        return hl + 1, vl                       # nhánh trái sâu hơn hoặc hòa

    return dfs(root)[1]

for vals, expected in examples:
    got = bottom_left_dp(build_tree(vals))
    assert got == expected
print("OK cách 4 trên ví dụ")

OK cách 4 trên ví dụ


In [15]:
class Solution:
    def findBottomLeftValue(self, root: Optional[TreeNode]) -> int:
        def dfs(node: Optional[TreeNode]):
            if node is None:
                return 0, None
            hl, vl = dfs(node.left)
            hr, vr = dfs(node.right)
            if hl == 0 and hr == 0:
                return 1, node.val
            if hr > hl:
                return hr + 1, vr
            return hl + 1, vl

        return dfs(root)[1]

print([Solution().findBottomLeftValue(build_tree(v)) for v, _ in examples])

[1, 7, 4]


**Độ phức tạp (cách 4):** Time `O(n)`; Space `O(h)`. Đây là mẫu **Tree DP**: đáp án của cha được ghép từ đáp án của hai con.

### Cách 5 — DFS lặp bằng ngăn xếp (không đệ quy)

**Ý tưởng:** làm đúng cách 3 nhưng thay đệ quy bằng ngăn xếp tường minh chứa `(node, depth)`.

Ngăn xếp lấy phần tử **vào sau ra trước**, nên để thăm **trái trước** ta phải đẩy **phải vào trước, trái vào sau**.

In [16]:
def bottom_left_dfs_iter(root):
    best_depth, ans = -1, None
    stack = [(root, 0)]
    while stack:
        node, depth = stack.pop()
        if depth > best_depth:
            best_depth, ans = depth, node.val
        if node.right:
            stack.append((node.right, depth + 1))      # phải vào trước
        if node.left:
            stack.append((node.left, depth + 1))       # trái vào sau => trái được lấy ra trước
    return ans

for vals, expected in examples:
    got = bottom_left_dfs_iter(build_tree(vals))
    assert got == expected
print("OK cách 5 trên ví dụ")

OK cách 5 trên ví dụ


In [17]:
class Solution:
    def findBottomLeftValue(self, root: Optional[TreeNode]) -> int:
        best_depth, ans = -1, root.val
        stack = [(root, 0)]
        while stack:
            node, depth = stack.pop()
            if depth > best_depth:
                best_depth, ans = depth, node.val
            if node.right:
                stack.append((node.right, depth + 1))
            if node.left:
                stack.append((node.left, depth + 1))
        return ans

print([Solution().findBottomLeftValue(build_tree(v)) for v, _ in examples])

[1, 7, 4]


**Độ phức tạp (cách 5):** Time `O(n)`; Space `O(h)` trường hợp thường (ngăn xếp chứa nhiều nhất cỡ chiều cao × 2). Không còn nguy cơ `RecursionError`.

## 8. TRACE từng bước

### 8.1. Trace cách 2 (BFS phải-trước-trái)

Hàm in từng lần `popleft`, hàng đợi trước/sau, và nút "hiện tại" (ứng viên đáp án).

| Cột | Ý nghĩa |
|---|---|
| `lấy ra` | nút vừa `popleft` (đây là ứng viên đáp án hiện tại) |
| `thêm` | các con được đưa vào hàng đợi (phải trước) |
| `queue` | nội dung hàng đợi sau bước đó |

In [18]:
def trace_bfs(root):
    queue = deque([root])
    node = root
    step = 0
    while queue:
        node = queue.popleft()
        added = []
        if node.right:
            queue.append(node.right)
            added.append(node.right.val)
        if node.left:
            queue.append(node.left)
            added.append(node.left.val)
        step += 1
        print(f"bước {step:>2}: lấy ra {node.val:>3} | thêm {added} | queue = {[x.val for x in queue]}")
    print("nút cuối cùng lấy ra =", node.val)
    return node.val

trace_bfs(build_tree([1, 2, 3, 4, None, 5, 6, None, None, 7]))

bước  1: lấy ra   1 | thêm [3, 2] | queue = [3, 2]
bước  2: lấy ra   3 | thêm [6, 5] | queue = [2, 6, 5]
bước  3: lấy ra   2 | thêm [4] | queue = [6, 5, 4]
bước  4: lấy ra   6 | thêm [] | queue = [5, 4]
bước  5: lấy ra   5 | thêm [7] | queue = [4, 7]
bước  6: lấy ra   4 | thêm [] | queue = [7]
bước  7: lấy ra   7 | thêm [] | queue = []
nút cuối cùng lấy ra = 7


7

Thứ tự lấy ra: `1, 3, 2, 6, 5, 4, 7`. Hàng 1 đi từ phải sang trái (`3, 2`), hàng 2 cũng vậy (`6, 5, 4`), hàng 3 chỉ có `7`. Nút cuối cùng là `7` — đáp án.

### 8.2. Trace cách 3 (DFS tiền tự theo dõi độ sâu)

Mỗi dòng in nút đang thăm, độ sâu, và quyết định cập nhật (`ghi nhận`) hay `bỏ qua`.

In [19]:
def trace_dfs(root):
    best_depth, ans = -1, None

    def dfs(node, depth):
        nonlocal best_depth, ans
        if node is None:
            return
        pad = "  " * depth
        if depth > best_depth:
            best_depth, ans = depth, node.val
            print(f"{pad}thăm {node.val} (độ sâu {depth}) -> GHI NHẬN: best_depth={best_depth}, ans={ans}")
        else:
            print(f"{pad}thăm {node.val} (độ sâu {depth}) -> bỏ qua (best_depth={best_depth})")
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)

    dfs(root, 0)
    print("đáp án:", ans)
    return ans

trace_dfs(build_tree([1, 2, 3, 4, None, 5, 6, None, None, 7]))

thăm 1 (độ sâu 0) -> GHI NHẬN: best_depth=0, ans=1
  thăm 2 (độ sâu 1) -> GHI NHẬN: best_depth=1, ans=2
    thăm 4 (độ sâu 2) -> GHI NHẬN: best_depth=2, ans=4
  thăm 3 (độ sâu 1) -> bỏ qua (best_depth=2)
    thăm 5 (độ sâu 2) -> bỏ qua (best_depth=2)
      thăm 7 (độ sâu 3) -> GHI NHẬN: best_depth=3, ans=7
    thăm 6 (độ sâu 2) -> bỏ qua (best_depth=3)
đáp án: 7


7

Chú ý nút `5` và `6` (độ sâu 2) **không** ghi đè vì `2 > 2` sai; chỉ `7` (độ sâu 3) mới ghi đè. Bây giờ trace ví dụ có đáp án là con **phải**: `[1,2,3,None,4,5]`.

In [20]:
trace_dfs(build_tree([1, 2, 3, None, 4, 5]))

thăm 1 (độ sâu 0) -> GHI NHẬN: best_depth=0, ans=1
  thăm 2 (độ sâu 1) -> GHI NHẬN: best_depth=1, ans=2
    thăm 4 (độ sâu 2) -> GHI NHẬN: best_depth=2, ans=4
  thăm 3 (độ sâu 1) -> bỏ qua (best_depth=2)
    thăm 5 (độ sâu 2) -> bỏ qua (best_depth=2)
đáp án: 4


4

### 8.3. Trace cách 4 (trả về `(chiều cao, giá trị)`)

In [21]:
def trace_dp(root):
    def dfs(node, depth):
        pad = "  " * depth
        if node is None:
            return 0, None
        hl, vl = dfs(node.left, depth + 1)
        hr, vr = dfs(node.right, depth + 1)
        if hl == 0 and hr == 0:
            res = (1, node.val)
        elif hr > hl:
            res = (hr + 1, vr)
        else:
            res = (hl + 1, vl)
        print(f"{pad}nút {node.val}: trái=({hl},{vl}) phải=({hr},{vr}) -> {res}")
        return res
    return dfs(root, 0)[1]

print("đáp án:", trace_dp(build_tree([1, 2, 3, 4, None, 5, 6, None, None, 7])))

    nút 4: trái=(0,None) phải=(0,None) -> (1, 4)
  nút 2: trái=(1,4) phải=(0,None) -> (2, 4)
      nút 7: trái=(0,None) phải=(0,None) -> (1, 7)
    nút 5: trái=(1,7) phải=(0,None) -> (2, 7)
    nút 6: trái=(0,None) phải=(0,None) -> (1, 6)
  nút 3: trái=(2,7) phải=(1,6) -> (3, 7)
nút 1: trái=(2,4) phải=(3,7) -> (4, 7)
đáp án: 7


## 9. Code nộp LeetCode

Chọn **cách 2 (BFS phải-trước-trái)**: ngắn nhất, khó sai nhất. Đây là code đúng class/method/signature (trên LeetCode `TreeNode`, `Optional`, `deque` đều có sẵn hoặc cần import; ở đây import để chạy tại máy):

In [22]:
from collections import deque
from typing import Optional


class Solution:
    def findBottomLeftValue(self, root: Optional[TreeNode]) -> int:
        queue = deque([root])
        node = root
        while queue:
            node = queue.popleft()
            if node.right:
                queue.append(node.right)
            if node.left:
                queue.append(node.left)
        return node.val


for vals, expected in examples:
    got = Solution().findBottomLeftValue(build_tree(vals))
    print(vals, "->", got)
    assert got == expected

[2, 1, 3] -> 1
[1, 2, 3, 4, None, 5, 6, None, None, 7] -> 7
[1, 2, 3, None, 4, 5] -> 4


## 10. Vì sao thuật toán đúng

**Cách 2 (BFS phải-trước-trái).** Quy nạp theo độ sâu: BFS thêm nút vào hàng đợi theo thứ tự cha được lấy ra, mỗi cha thêm con **phải trước, trái sau**. Nếu các nút ở hàng `d` được lấy ra theo thứ tự từ phải sang trái thì các con của chúng được thêm vào theo thứ tự: con phải và con trái của nút ngoài cùng phải trước, ..., nghĩa là các nút hàng `d + 1` cũng sẽ được lấy ra từ phải sang trái. Hàng đợi là FIFO nên hàng `d` luôn được lấy hết trước hàng `d + 1`. Như vậy nút lấy ra **cuối cùng** là nút ngoài cùng **bên trái** của hàng **sâu nhất**. Đó đúng là đáp án.

**Cách 3 (DFS tiền tự).** Gọi `D` là độ sâu lớn nhất. Trong tiền tự "trái trước", với mỗi độ sâu `d`, nút trái nhất của độ sâu đó được thăm **trước mọi nút khác cùng độ sâu**. (Vì nếu `u` ở bên trái `v` cùng độ sâu thì tổ tiên chung thấp nhất `a` của chúng có `u` ở cây con trái và `v` ở cây con phải, và DFS thăm cây con trái của `a` trước cây con phải.) Biến `best_depth` tăng đúng khi gặp nút đầu tiên của một độ sâu mới; vì điều kiện là `>` chứ không phải `>=`, các nút khác cùng độ sâu không ghi đè. Cuối cùng `ans` là nút đầu tiên chạm độ sâu `D` — trái nhất của hàng cuối.

**Cách 4 (Tree DP).** Giả thiết quy nạp: `dfs(child)` trả về chiều cao và nút trái nhất của hàng cuối trong cây con. Hàng cuối của cây con gốc `node` nằm sâu nhất; nếu một nhánh sâu hơn hẳn thì hàng cuối chỉ nằm ở nhánh đó. Nếu hòa thì hàng cuối gồm các nút ở cả hai nhánh, và nút trái nhất thuộc nhánh **trái** (mọi nút nhánh trái ở bên trái mọi nút nhánh phải cùng độ sâu).

**Cách 5** giống cách 3, chỉ khác cách duyệt, và vì đẩy phải trước trái sau nên thứ tự thăm vẫn là tiền tự "trái trước".

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `h` là chiều cao, `w` là độ rộng lớn nhất.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê mọi hàng | `O(n)` | `O(n)` |
| 2. BFS phải-trước-trái | `O(n)` | `O(w)` |
| 3. DFS đệ quy + độ sâu | `O(n)` | `O(h)` |
| 4. DFS Tree DP | `O(n)` | `O(h)` |
| 5. DFS lặp (ngăn xếp) | `O(n)` | `O(h)` |

Mọi cách đều phải thăm **mọi** nút ít nhất một lần (không thể biết hàng cuối mà không nhìn hết cây), nên `O(n)` là tối ưu về thời gian. Với `n = 10^4` mọi cách đều chạy tức thì.

## 12. Bộ test

### 12.1. Ví dụ đề và edge case

Ta gom các cách vào dict `solvers` để test chung. Oracle độc lập: gán mỗi nút một cặp `(độ sâu, vị trí)`, với vị trí đánh số kiểu "heap" (con trái `2p`, con phải `2p+1`), rồi lấy nút có độ sâu lớn nhất và vị trí nhỏ nhất.

In [23]:
def oracle(root):
    """Độc lập: (độ sâu, vị trí heap) cho mọi nút; chọn sâu nhất, rồi vị trí nhỏ nhất."""
    best = None
    stack = [(root, 0, 1)]
    while stack:
        node, depth, pos = stack.pop()
        key = (-depth, pos)                       # sâu hơn -> nhỏ hơn; cùng sâu -> vị trí nhỏ hơn
        if best is None or key < best[0]:
            best = (key, node.val)
        if node.left:
            stack.append((node.left, depth + 1, 2 * pos))
        if node.right:
            stack.append((node.right, depth + 1, 2 * pos + 1))
    return best[1]

solvers = {
    "levels": bottom_left_levels,
    "bfs": bottom_left_bfs,
    "dfs": bottom_left_dfs,
    "dp": bottom_left_dp,
    "dfs_iter": bottom_left_dfs_iter,
    "leetcode": lambda r: Solution().findBottomLeftValue(r),
}

for vals, expected in examples:
    root = build_tree(vals)
    assert oracle(root) == expected
    for name, fn in solvers.items():
        assert fn(root) == expected, (name, vals)
print("Ví dụ đề: OK cho cả 6 cách và oracle")

Ví dụ đề: OK cho cả 6 cách và oracle


In [24]:
edge_cases = [
    ([5], 5, "một nút"),
    ([-7], -7, "một nút âm"),
    ([2**31 - 1, -2**31], -2**31, "biên giá trị, một con trái"),
    ([1, None, 2], 2, "một con phải"),
    ([1, 2], 2, "một con trái"),
    ([1, 2, 3], 2, "hàng cuối [2,3]"),
    ([1, 2, 3, 4, 5, 6, 7], 4, "cây đầy đủ"),
    ([1, 2, 3, None, 4, 5], 4, "đáp án là con phải của cha"),
    ([1, 2, 3, 4, None, None, 5, 6], 6, "nhánh trái sâu hơn"),
    ([1, 2, 3, None, None, 4, None, None, 5], 5, "nhánh phải sâu hơn"),
    ([1, 2, 3, 4, None, 5, 6, None, None, 7], 7, "ví dụ 2"),
    ([0, 0, 0, 0, 0, 0, 0], 0, "toàn số 0"),
]
for vals, expected, note in edge_cases:
    root = build_tree(vals)
    assert oracle(root) == expected, ("oracle", note)
    for name, fn in solvers.items():
        assert fn(root) == expected, (name, note)
    print("OK:", note, "->", expected)

OK: một nút -> 5
OK: một nút âm -> -7
OK: biên giá trị, một con trái -> -2147483648
OK: một con phải -> 2
OK: một con trái -> 2
OK: hàng cuối [2,3] -> 2
OK: cây đầy đủ -> 4
OK: đáp án là con phải của cha -> 4
OK: nhánh trái sâu hơn -> 6
OK: nhánh phải sâu hơn -> 5
OK: ví dụ 2 -> 7
OK: toàn số 0 -> 0


### 12.2. Phản ví dụ cho các cách làm sai

**Sai 1 — chỉ đi theo `left` mãi.** **Sai 2 — DFS dùng `>=`** (ra nút phải nhất của hàng cuối). **Sai 3 — BFS trái-trước, lấy nút cuối** (ra nút phải nhất). **Sai 4 — khởi tạo `best_depth = 0`** (bỏ sót gốc ở cây một nút).

In [25]:
def wrong_left_only(root):
    node = root
    while node.left:
        node = node.left
    return node.val

def wrong_ge(root):
    best_depth, ans = -1, None
    def dfs(node, depth):
        nonlocal best_depth, ans
        if node is None:
            return
        if depth >= best_depth:                     # SAI: >=
            best_depth, ans = depth, node.val
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)
    dfs(root, 0)
    return ans

def wrong_bfs_left_first(root):
    queue = deque([root])
    node = root
    while queue:
        node = queue.popleft()
        if node.left:                               # SAI: trái trước -> nút cuối là PHẢI nhất
            queue.append(node.left)
        if node.right:
            queue.append(node.right)
    return node.val

def wrong_depth_zero(root):
    best_depth, ans = 0, None                       # SAI: gốc có độ sâu 0 nên không bao giờ > 0
    def dfs(node, depth):
        nonlocal best_depth, ans
        if node is None:
            return
        if depth > best_depth:
            best_depth, ans = depth, node.val
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)
    dfs(root, 0)
    return ans

all_cases = [v for v, _ in examples] + [v for v, _, _ in edge_cases]
for fn in (wrong_left_only, wrong_ge, wrong_bfs_left_first, wrong_depth_zero):
    bad = [v for v in all_cases if fn(build_tree(v)) != oracle(build_tree(v))]
    print(fn.__name__, "sai trên", len(bad), "test, ví dụ:", bad[0])
    assert bad

wrong_left_only sai trên 6 test, ví dụ: [1, 2, 3, 4, None, 5, 6, None, None, 7]
wrong_ge sai trên 5 test, ví dụ: [2, 1, 3]
wrong_bfs_left_first sai trên 5 test, ví dụ: [2, 1, 3]
wrong_depth_zero sai trên 2 test, ví dụ: [5]


### 12.3. Đối chiếu ngẫu nhiên (seed cố định) đúng ràng buộc

Sinh cây ngẫu nhiên `1 <= n <= 80`, giá trị trong `[-2^31, 2^31 - 1]` hoặc khoảng nhỏ (để có nhiều giá trị trùng), so tất cả cách với oracle.

In [26]:
def random_tree_list(rng, n, lo, hi):
    """Sinh cây ngẫu nhiên n nút (n >= 1) với giá trị trong [lo, hi], trả về list kiểu LeetCode."""
    root = TreeNode(rng.randint(lo, hi))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        k = rng.randrange(len(slots))
        slots[k], slots[-1] = slots[-1], slots[k]
        parent, side = slots.pop()
        child = TreeNode(rng.randint(lo, hi))
        setattr(parent, side, child)
        slots.append((child, "left"))
        slots.append((child, "right"))
    return tree_to_list(root)


rng = random.Random(513)
checked = 0
for _ in range(1500):
    n = rng.randint(1, 80)
    lo, hi = rng.choice([(-3, 3), (-1000, 1000), (-2**31, 2**31 - 1)])
    vals = random_tree_list(rng, n, lo, hi)
    root = build_tree(vals)
    want = oracle(root)
    for name, fn in solvers.items():
        assert fn(root) == want, (name, vals)
    checked += 1
print("đã đối chiếu", checked, "cây ngẫu nhiên: 6 cách khớp oracle")

đã đối chiếu 1500 cây ngẫu nhiên: 6 cách khớp oracle


In [27]:
# Hai kiểu cây cực đoan tối đa 10^4 nút: lệch trái, lệch phải, zig-zag
def chain_tree(n, side):
    root = TreeNode(0)
    cur = root
    for i in range(1, n):
        nxt = TreeNode(i)
        if side == "left":
            cur.left = nxt
        elif side == "right":
            cur.right = nxt
        else:                                       # zig-zag
            if i % 2:
                cur.left = nxt
            else:
                cur.right = nxt
        cur = nxt
    return root

n = 10000
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(50000)
for side in ("left", "right", "zigzag"):
    root = chain_tree(n, side)
    for name, fn in solvers.items():
        assert fn(root) == n - 1, (side, name)
    print(side, "OK: đáp án", n - 1)
sys.setrecursionlimit(old_limit)

left OK: đáp án 9999
right OK: đáp án 9999
zigzag OK: đáp án 9999


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ đi theo `left` mãi

**Code sai:**

In [28]:
class WrongSolution1:
    def findBottomLeftValue(self, root):
        while root.left:
            root = root.left
        return root.val

t = build_tree([1, 2, 3, 4, None, 5, 6, None, None, 7])
print("sai :", WrongSolution1().findBottomLeftValue(t))
print("đúng:", Solution().findBottomLeftValue(t))

sai : 4
đúng: 7


**Vì sao sai:** nhánh trái có thể **nông hơn** nhánh phải. Hàng cuối của cây là hàng sâu nhất, có thể nằm hoàn toàn ở nhánh phải; và nút trái nhất của hàng đó đôi khi là con **phải** của cha.

**Code đúng:** BFS phải-trước-trái (cách 2) hoặc DFS có theo dõi độ sâu (cách 3).

### Lỗi 2: Dùng `>=` khi cập nhật độ sâu trong DFS

In [29]:
class WrongSolution2:
    def findBottomLeftValue(self, root):
        best_depth, ans = -1, None
        def dfs(node, depth):
            nonlocal best_depth, ans
            if node is None:
                return
            if depth >= best_depth:        # SAI: nút phải cùng độ sâu sẽ ghi đè
                best_depth, ans = depth, node.val
            dfs(node.left, depth + 1)
            dfs(node.right, depth + 1)
        dfs(root, 0)
        return ans

t = build_tree([1, 2, 3])
print("sai :", WrongSolution2().findBottomLeftValue(t), "(ra nút phải nhất)")
print("đúng:", Solution().findBottomLeftValue(t))

sai : 3 (ra nút phải nhất)
đúng: 2


**Vì sao sai:** với `>=`, mỗi nút **cùng độ sâu** gặp sau (tức bên phải) sẽ ghi đè nút trái. Chỉ dùng `>` để nút **đầu tiên** của mỗi độ sâu "giữ chỗ".

### Lỗi 3: BFS thêm trái trước rồi lấy nút cuối

In [30]:
t = build_tree([1, 2, 3, 4, 5, 6, 7])
print("sai :", wrong_bfs_left_first(t), "(nút cuối của hàng cuối = nút PHẢI nhất)")
print("đúng:", Solution().findBottomLeftValue(t))

sai : 7 (nút cuối của hàng cuối = nút PHẢI nhất)
đúng: 4


**Vì sao sai:** BFS "thường" (trái trước) lấy ra hàng cuối theo thứ tự trái → phải, nên nút **cuối cùng** là phải nhất. Muốn dùng mẹo "nút cuối cùng", phải thêm **phải trước**. Hoặc dùng BFS thường và lấy nút **đầu tiên** của hàng cuối (cần biết ranh giới hàng, cách 1).

### Lỗi 4: Khởi tạo `best_depth = 0` (bỏ sót cây một nút)

In [31]:
t = build_tree([42])
print("sai :", wrong_depth_zero(t))
print("đúng:", Solution().findBottomLeftValue(t))

sai : None
đúng: 42


**Vì sao sai:** gốc có độ sâu `0`; điều kiện `depth > best_depth` với `best_depth = 0` không bao giờ đúng cho gốc, nên `ans` vẫn là `None`. Khởi tạo `best_depth = -1` hoặc gán sẵn `ans = root.val`.

### Lỗi 5: Quên chốt kích thước hàng khi BFS theo tầng

In [32]:
def wrong_levels_no_size(root):
    queue = deque([root])
    first_of_level = root.val
    while queue:
        # SAI: không chốt size, nên không biết đâu là ranh giới hàng
        node = queue.popleft()
        first_of_level = node.val        # luôn là nút mới nhất => nút cuối cùng (không phải đầu hàng)
        if node.left:
            queue.append(node.left)
        if node.right:
            queue.append(node.right)
    return first_of_level

t = build_tree([1, 2, 3, 4, 5, 6, 7])
print("sai :", wrong_levels_no_size(t))
print("đúng:", Solution().findBottomLeftValue(t))

sai : 7
đúng: 4


**Vì sao sai:** nếu muốn lấy **phần tử đầu của mỗi hàng** thì phải biết hàng nào bắt đầu ở đâu — dùng `for _ in range(len(queue))` (chốt `size` trước) như ở mục 3.3 và cách 1.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Hàng cuối / hàng sâu nhất / theo từng tầng"** → nghĩ ngay **BFS theo tầng** (hoặc DFS có `depth`).
- Muốn "nút trái nhất ở hàng sâu nhất" mà code ngắn nhất: **BFS phải-trước-trái** rồi lấy nút cuối.
- DFS tiền tự **trái trước** + `depth > best_depth` (dấu `>` chặt) = "nút đầu tiên của độ sâu mới".
- Hãy kiểm tra tay trên hai kiểu cây: (1) nhánh trái nông hơn nhánh phải; (2) đáp án là con **phải** của cha.
- Quy ước độ sâu: gốc = `0` hay `1` — chọn một và khởi tạo `best_depth` phù hợp (`-1` hoặc `0`).
- Cây lệch `10^4` nút → cân nhắc BFS hoặc DFS lặp để tránh đệ quy sâu.

## 15. Biến thể

1. **Rightmost value ở hàng cuối:** đối xứng (BFS trái-trước-phải, lấy nút cuối; hoặc DFS phải trước).
2. **Right Side View (199) / Left Side View:** lấy nút ngoài cùng mỗi hàng.
3. **Find Largest Value in Each Tree Row (515):** thay "phần tử đầu" bằng `max` của hàng.
4. **Deepest Leaves Sum (1302):** tổng các nút ở hàng cuối.
5. **Nút trái nhất ở độ sâu `k`:** dừng sớm khi tới hàng `k`.
6. **Cây N-ary:** thay `left/right` bằng `children` (duyệt con theo thứ tự ngược để lấy nút cuối).

Ví dụ biến thể 2 (left side view) và 4 (tổng hàng cuối):

In [33]:
def left_side_view(root):
    view = []
    queue = deque([root])
    while queue:
        view.append(queue[0].val)                      # phần tử đầu của hàng
        for _ in range(len(queue)):
            node = queue.popleft()
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
    return view

def deepest_leaves_sum(root):
    queue = deque([root])
    total = 0
    while queue:
        total = 0
        for _ in range(len(queue)):
            node = queue.popleft()
            total += node.val
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
    return total

t = build_tree([1, 2, 3, 4, None, 5, 6, None, None, 7])
print(left_side_view(t), deepest_leaves_sum(t))
print(left_side_view(build_tree([1, 2, 3, None, 4, 5])), deepest_leaves_sum(build_tree([1, 2, 3, None, 4, 5])))

[1, 2, 4, 7] 7
[1, 2, 4] 9


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê mọi hàng | BFS theo tầng, lấy `levels[-1][0]` | `O(n)` | `O(n)` | Dễ hiểu nhất, đối chiếu |
| 2. BFS phải-trước-trái | Nút cuối cùng lấy ra là đáp án | `O(n)` | `O(w)` | **Code ngắn nhất, nên nộp** |
| 3. DFS + độ sâu | Nút đầu tiên chạm độ sâu mới (`>`) | `O(n)` | `O(h)` | Khi quen DFS, ít bộ nhớ trên cây rộng |
| 4. DFS Tree DP | Trả `(chiều cao, giá trị)`, hòa chọn trái | `O(n)` | `O(h)` | Dạng bài ghép đáp án con -> cha |
| 5. DFS lặp (ngăn xếp) | Giống 3 nhưng không đệ quy | `O(n)` | `O(h)` | Cây rất sâu (tránh `RecursionError`) |

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán đầu ra rồi chạy:

In [34]:
print(Solution().findBottomLeftValue(build_tree([2, 1, 3])))
print(Solution().findBottomLeftValue(build_tree([1, 2, 3, None, 4, 5])))

1
4


**Bài 2 (tính tay).** Cây `[3,9,20,None,None,15,7]`: viết ra các hàng, rồi cho biết đáp án. Gán vào `ans2` (đang là `None`).

In [35]:
ans2 = None      # TODO: điền số nguyên

**Bài 3 (điền TODO).** Viết `levels_of(root)` trả về **list các list** giá trị theo từng hàng (từ trái sang phải). Ví dụ `[3,9,20,None,None,15,7]` -> `[[3],[9,20],[15,7]]`.

In [36]:
def levels_of(root):
    # TODO: BFS theo tầng, chốt size = len(queue) ở đầu mỗi hàng
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_bottom_left(root)` theo cách 3 (DFS có `depth`, `best_depth`).

In [37]:
def my_bottom_left(root):
    # TODO
    pass

**Bài 5 (mini challenge).** Viết `leftmost_at_depth(root, k)` trả về giá trị nút trái nhất ở độ sâu `k` (gốc có độ sâu `0`), hoặc `None` nếu cây không có độ sâu `k`. Sau đó viết `bottom_right_value(root)` (nút **phải** nhất ở hàng cuối).

In [38]:
def leftmost_at_depth(root, k):
    # TODO
    pass

def bottom_right_value(root):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: viết ra các hàng của cây.
- Bài 2: liệt kê từng hàng; hàng cuối là hàng nào?
- Bài 3: BFS, mỗi vòng ngoài xử lý đúng một hàng.
- Bài 4: nhớ truyền `depth + 1` xuống con.
- Bài 5: đi tới độ sâu `k` rồi dừng.

**Mức 2 (vừa):**
- Bài 2: hàng cuối `[15, 7]`, phần tử đầu là đáp án.
- Bài 3: `for _ in range(len(queue))` rồi `row.append(node.val)`; sau vòng for, `levels.append(row)`.
- Bài 4: `best_depth = -1`; `if depth > best_depth`; gọi `left` trước.
- Bài 5: `leftmost_at_depth` = phần tử đầu của `levels_of(root)[k]` nếu có; `bottom_right_value` = phần tử cuối của hàng cuối.

**Mức 3 (gần đáp án):**
- Bài 3: xem mục 3.3 (hàm `bfs_levels_demo`), chỉ cần thu `row` vào `levels`.
- Bài 4: xem cách 3 ở mục 7.
- Bài 5: `levels = levels_of(root)`; `return levels[k][0] if k < len(levels) else None`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [39]:
# Bài 1: 1 và 4
assert Solution().findBottomLeftValue(build_tree([2, 1, 3])) == 1
assert Solution().findBottomLeftValue(build_tree([1, 2, 3, None, 4, 5])) == 4

# Bài 2: các hàng [3], [9,20], [15,7] -> hàng cuối [15,7] -> 15
ans2 = 15
assert ans2 == Solution().findBottomLeftValue(build_tree([3, 9, 20, None, None, 15, 7]))
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [40]:
# Bài 3
def levels_of(root):
    levels = []
    queue = deque([root])
    while queue:
        row = []
        for _ in range(len(queue)):
            node = queue.popleft()
            row.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        levels.append(row)
    return levels

assert levels_of(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [9, 20], [15, 7]]
assert levels_of(build_tree([1, 2, 3, 4, None, 5, 6, None, None, 7])) == [[1], [2, 3], [4, 5, 6], [7]]
assert levels_of(build_tree([5])) == [[5]]
print("Bài 3 OK")

Bài 3 OK


In [41]:
# Bài 4
def my_bottom_left(root):
    best_depth, ans = -1, None
    def dfs(node, depth):
        nonlocal best_depth, ans
        if node is None:
            return
        if depth > best_depth:
            best_depth, ans = depth, node.val
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)
    dfs(root, 0)
    return ans

rng = random.Random(3)
for _ in range(200):
    root = build_tree(random_tree_list(rng, rng.randint(1, 40), -50, 50))
    assert my_bottom_left(root) == oracle(root)
print("Bài 4 OK")

Bài 4 OK


In [42]:
# Bài 5
def leftmost_at_depth(root, k):
    levels = levels_of(root)
    return levels[k][0] if 0 <= k < len(levels) else None

def bottom_right_value(root):
    return levels_of(root)[-1][-1]

t = build_tree([1, 2, 3, 4, None, 5, 6, None, None, 7])
assert [leftmost_at_depth(t, k) for k in range(5)] == [1, 2, 4, 7, None]
assert bottom_right_value(t) == 7
t2 = build_tree([1, 2, 3, 4, 5, 6, 7])
assert bottom_right_value(t2) == 7 and leftmost_at_depth(t2, 2) == 4
assert bottom_right_value(build_tree([1, 2, 3, None, 4, 5])) == 5
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template BFS phải-trước-trái (nút cuối cùng lấy ra là đáp án):**

```python
from collections import deque

class Solution:
    def findBottomLeftValue(self, root):
        queue = deque([root])
        node = root
        while queue:
            node = queue.popleft()
            if node.right:
                queue.append(node.right)     # PHẢI trước
            if node.left:
                queue.append(node.left)
        return node.val
```

**Template DFS (nút đầu tiên chạm độ sâu mới):**

```python
best_depth, ans = -1, root.val
def dfs(node, depth):
    nonlocal best_depth, ans
    if not node:
        return
    if depth > best_depth:          # dấu > chặt
        best_depth, ans = depth, node.val
    dfs(node.left, depth + 1)       # TRÁI trước
    dfs(node.right, depth + 1)
```

**Checklist trước khi submit:**

- [ ] "Hàng cuối" = độ sâu **lớn nhất**, không phải đi mãi về trái?
- [ ] BFS: đã thêm **phải trước, trái sau** (nếu dùng mẹo nút cuối cùng)?
- [ ] DFS: duyệt **trái trước** và so sánh `depth > best_depth` (chặt)?
- [ ] `best_depth` khởi tạo `-1` (hoặc gán sẵn `ans = root.val`) cho cây một nút?
- [ ] Đã thử: một nút, nhánh trái nông hơn phải, đáp án là con phải của cha?
- [ ] Cây lệch `10^4` nút có gây `RecursionError`? (dùng cách lặp nếu cần)